# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) trên Kaggle
Đọc README.md và GUIDE.md trước. Bản Kaggle giữ nguyên rubric/cách chia dữ liệu của repo.

## 0. Môi trường Kaggle GPU
Bật Internet và chọn GPU trong Notebook options. Thêm dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` qua Add Input trước khi chạy tất cả.

In [ ]:
# Clone repo công khai để dùng scripts/ và cấu hình chính thức của đề.
%cd /kaggle/working
!if [ ! -d lab/.git ]; then git clone --depth 1 https://github.com/nvchien19/K4-Track-4-Day10-VisionLanguageModel.git lab; fi
%cd /kaggle/working/lab
!pip -q install -r requirements-colab.txt
!nvidia-smi -L

In [ ]:
from pathlib import Path
import zipfile

input_root = Path("/kaggle/input")
extract_root = Path("/kaggle/working/dataset_unpacked")
def find_dataset_root(base):
    for p in [base, *base.rglob("*")]:
        if p.is_dir() and (p / "images").is_dir() and (p / "labels-YOLO").is_dir():
            return p
    return None

dataset_root = find_dataset_root(input_root)
if dataset_root is None:
    zip_files = list(input_root.rglob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("Hãy Add Input dataset road-damage-dataset-potholes-cracks-and-manholes trước khi chạy.")
    extract_root.mkdir(parents=True, exist_ok=True)
    for archive in zip_files:
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(extract_root)
    dataset_root = find_dataset_root(extract_root)
if dataset_root is None:
    raise FileNotFoundError("Không tìm thấy images/ và labels-YOLO/ trong Kaggle Input.")
print("Dataset source:", dataset_root)

## 1. Dữ liệu: tải và chia split

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", str(dataset_root)], check=True)

In [ ]:
# Kiểm tra split và trực quan hoá 6 ảnh (hộp màu theo lớp).
import json, random
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
for name, rows in splits.items():
    print(name, "images=", len(rows), "boxes=", sum(len(r["objects"]) for r in rows))
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sessions["train"] & sessions["val"] or sessions["train"] & sessions["test"] or sessions["val"] & sessions["test"])
print("dataset_version:", meta["dataset_version"], "; session leakage: none")

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "lime"}
for row in random.Random(42).sample(splits["train"], min(6, len(splits["train"]))):
    sample_path = Path(row["image_path"])
    if not sample_path.exists(): sample_path = Path("data") / sample_path
    image = Image.open(sample_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    for obj in row["objects"]:
        xc, yc, w, h = obj["yolo"]
        box = ((xc-w/2)*row["width"], (yc-h/2)*row["height"], (xc+w/2)*row["width"], (yc+h/2)*row["height"])
        draw.rectangle(box, outline=COLORS[obj["label"]], width=2)
        draw.text((box[0], max(0, box[1]-12)), obj["label"], fill=COLORS[obj["label"]])
    print(row["image_id"]); display(image)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json, math, os, re, time, random, contextlib
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
CLASSES = ("pothole", "crack", "manhole")
PROMPT = "Detect all road damages in this image. Classes: pothole, crack, manhole. Return only a JSON list, each item with label and bbox_2d [x1,y1,x2,y2] as integers on the 0-1000 coordinate scale relative to image width and height. Return [] if none."
PIXEL_PROMPT = "Detect all road damages in this image. Classes: pothole, crack, manhole. Return only a JSON list, each item with label and bbox_2d [x1,y1,x2,y2] as pixel integers relative to the 640x360 image. Return [] if none."

def yolo_to_box(label, yolo, width=640, height=360, coord="norm1000"):
    xc, yc, w, h = map(float, yolo)
    scale_x, scale_y = ((1000, 1000) if coord == "norm1000" else (width, height))
    vals = ((xc-w/2)*scale_x, (yc-h/2)*scale_y, (xc+w/2)*scale_x, (yc+h/2)*scale_y)
    x1, y1, x2, y2 = [int(max(0, min(limit, round(v)))) for v, limit in zip(vals, (scale_x, scale_y, scale_x, scale_y))]
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects, coord="norm1000"):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord=coord) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

print(objects_to_answer(splits["train"][0]["objects"]))


## 3. E0: zero-shot

In [ ]:
# Khởi tạo model gốc 4-bit; cấu hình NF4 trên Kaggle.

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True, llm_int8_skip_modules=["model.visual", "lm_head"])

processor = AutoProcessor.from_pretrained(MODEL_ID)

processor.tokenizer.padding_side = "left"

device = torch.device("cuda:0")



def load_base():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16,
        device_map={"": 0})
    import bitsandbytes as bnb
    assert not any(isinstance(x, bnb.nn.Linear4bit) for x in model.model.visual.modules()), "visual must remain dense"
    model.config.use_cache = False
    return model
def image_path(row):

    path = Path(row["image_path"])

    return path if path.exists() else Path("data") / path



@torch.inference_mode()

def predict_rows(model, rows, prompt=PROMPT, batch_size=1, max_new_tokens=256):

    model.eval(); output = []

    for start in range(0, len(rows), batch_size):

        batch = rows[start:start+batch_size]

        messages = [[{"role":"user", "content":[{"type":"image", "image":Image.open(image_path(r)).convert("RGB")}, {"type":"text", "text":prompt}]}] for r in batch]

        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,

                    return_dict=True, return_tensors="pt", padding=True).to(device)

        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

        prompt_len = inputs["input_ids"].shape[1]

        decoded = processor.batch_decode(generated[:, prompt_len:], skip_special_tokens=True)

        output.extend({"image_id": r["image_id"], "raw": raw.strip()} for r, raw in zip(batch, decoded))

    return output



def write_jsonl(path, rows):

    Path(path).parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:

        for row in rows: f.write(json.dumps(row, ensure_ascii=False) + "\n")





In [ ]:
# E0: chạy zero-shot trên test sau khi mọi hàm đã được định nghĩa.
model_e0 = load_base()
e0_preds = predict_rows(model_e0, splits["test"], batch_size=1)
write_jsonl("work/e0.jsonl", e0_preds)
print("E0 predictions:", len(e0_preds))
del model_e0, e0_preds
torch.cuda.empty_cache()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
def make_messages(row, coord="norm1000"):
    prompt = PROMPT if coord == "norm1000" else PIXEL_PROMPT
    image = Image.open(image_path(row)).convert("RGB")
    user = {"role":"user", "content":[{"type":"image", "image":image}, {"type":"text", "text":prompt}]}
    return user, {"role":"assistant", "content":objects_to_answer(row["objects"], coord=coord)}

def encode_example(row, coord="norm1000"):
    user, answer = make_messages(row, coord)
    prompt_enc = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    full_enc = processor.apply_chat_template([user, answer], tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), "prompt tokens are not a prefix; inspect chat template"
    labels = full_enc["input_ids"].clone(); labels[:, :n] = -100
    full_enc["labels"] = labels
    return full_enc

def collate(examples):
    pad_id = processor.tokenizer.pad_token_id
    max_len = max(x["input_ids"].shape[1] for x in examples)
    keys = ("input_ids", "attention_mask", "mm_token_type_ids", "labels")
    out = {}
    for key in keys:
        vals = [x[key] for x in examples if key in x]
        if not vals: continue
        fill = -100 if key == "labels" else (pad_id if key == "input_ids" else 0)
        padded = []
        for value in vals:
            amount = max_len - value.shape[1]
            padded.append(torch.nn.functional.pad(value, (0, amount), value=fill))
        out[key] = torch.cat(padded, dim=0)
    for key in ("pixel_values", "image_grid_thw"):
        vals = [x[key] for x in examples if key in x]
        if vals: out[key] = torch.cat(vals, dim=0)
    return out

sample = encode_example(splits["train"][0])
print("image patches/grid:", sample.get("image_grid_thw"), "supervised tokens:", int((sample["labels"] != -100).sum()))
assert "pixel_values" in sample and "image_grid_thw" in sample


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
def apply_setup(model, setup, rank=16):
    for p in model.parameters(): p.requires_grad_(False)
    model.enable_input_require_grads()
    llm = r"model\.language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
    vit = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))"
    target = llm if setup in ("E1", "E2") else f"({llm})|({vit})"
    names = [n for n, _ in model.named_modules() if re.fullmatch(target, n)]
    expected = 252 if setup in ("E1", "E2") else 348
    assert len(names) == expected, f"LoRA target count {len(names)} != {expected}; inspect model.named_modules()"
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.05,
                          target_modules=target, task_type="CAUSAL_LM"))
    if setup in ("E2", "E3"):
        found = 0
        for name, param in model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.float(); param.requires_grad_(True); found += param.numel()
        assert found >= 100_000_000, f"projector trainable size unexpectedly small: {found}"
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(setup, "matched modules=", len(names), "trainable params=", f"{trainable:,}")
    return model, trainable


## 6. Vòng huấn luyện

In [ ]:
def train_experiment(setup, rank=16, train_fraction=1.0, max_steps=300, seed=42):

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    rows = list(splits["train"])

    random.Random(seed).shuffle(rows)

    rows = rows[:max(1, int(len(rows)*train_fraction))]

    model, trainable = apply_setup(load_base(), setup, rank=rank)

    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})

    model.config.use_cache = False

    train_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(train_params, lr=(2e-5 if setup in ("E2", "E3") else 2e-4), weight_decay=0, foreach=False)

    loader = DataLoader(rows, batch_size=1, shuffle=True, collate_fn=lambda batch: collate([encode_example(r) for r in batch]), drop_last=True)

    total = max_steps

    warmup = max(1, int(total * 0.05))

    def lr_lambda(step):

        if step < warmup: return (step + 1) / warmup

        progress = min(1.0, (step - warmup) / max(1, total - warmup))

        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scaler = torch.amp.GradScaler("cuda")

    optimizer.zero_grad(set_to_none=True); step = 0; micro = 0; losses = []; started = time.time()

    torch.cuda.reset_peak_memory_stats()

    run_dir = Path(f"work/{setup.lower()}_r{rank}"); run_dir.mkdir(parents=True, exist_ok=True)

    while step < max_steps:

        for batch in loader:

            batch = {k:v.to(device) for k,v in batch.items()}

            with torch.autocast("cuda", dtype=torch.float16): loss = model(**batch).loss / 8

            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite loss at optimizer step {step}")

            scaler.scale(loss).backward(); losses.append(float(loss.detach())*8); micro += 1

            if micro == 8:

                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(train_params, 1.0)

                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()

                step += 1; micro = 0

                if step % 25 == 0: print(setup, step, "loss=", round(sum(losses[-100:])/len(losses[-100:]), 4))

                if step % 25 == 0 or step == max_steps:
                    save_experiment(model, run_dir)
                    (run_dir/"progress.json").write_text(json.dumps({"step":step,"max_steps":max_steps,"mean_last_loss":sum(losses[-100:])/len(losses[-100:])}), encoding="utf-8")

                if step >= max_steps: break

    minutes = (time.time()-started)/60

    run = {"setup":setup,"rank":rank,"trainable_params":trainable,"peak_vram_gb":torch.cuda.max_memory_allocated()/1024**3,"train_minutes":minutes,"max_steps":step,"mean_last_loss":sum(losses[-100:])/len(losses[-100:])}

    (run_dir/"run.json").write_text(json.dumps(run, indent=2), encoding="utf-8")

    print(run)

    return run_dir, run



def save_experiment(model, run_dir):

    model.save_pretrained(run_dir / "adapter")

    extra = {n:p.detach().float().cpu() for n,p in model.named_parameters() if p.requires_grad and ("visual.merger" in n or "visual.deepstack_merger_list" in n)}

    if extra: torch.save(extra, run_dir / "extra_trainable.pt")



def train_all():

    runs = {}

    for setup in ("E1", "E2", "E3"):

        runs[setup] = train_experiment(setup)

    return runs



# Chạy sau khi xác nhận GPU/dữ liệu sẵn sàng; mỗi thí nghiệm khoảng 20–30 phút theo đề.


## 7. Suy luận và chấm E1–E3

In [ ]:
def load_trained(run_dir, setup):
    model = load_base()
    model = PeftModel.from_pretrained(model, Path(run_dir)/"adapter", is_trainable=False)
    extra_path = Path(run_dir)/"extra_trainable.pt"
    if extra_path.exists():
        state = torch.load(extra_path, map_location="cpu", weights_only=True)
        missing, unexpected = model.load_state_dict(state, strict=False)
        assert not unexpected, unexpected
        assert not [n for n in missing if "visual.merger" in n or "visual.deepstack_merger_list" in n]
    return model

def score_run(run_dir, setup, evidence_name=None, coord="norm1000"):
    run = json.loads((Path(run_dir)/"run.json").read_text(encoding="utf-8"))
    model = load_trained(run_dir, setup)
    preds = predict_rows(model, splits["test"], prompt=PROMPT if coord == "norm1000" else PIXEL_PROMPT)
    pred_path = f"work/{setup.lower()}.jsonl"; write_jsonl(pred_path, preds)
    out = evidence_name or f"report/evidence/{setup.lower()}.json"
    meta_args = f"--meta trainable_params={run['trainable_params']} --meta peak_vram_gb={run['peak_vram_gb']:.5f} --meta train_minutes={run['train_minutes']:.5f}"
    coord_arg = " --coord pixel" if coord == "pixel" else ""
    !python scripts/score.py --pred {pred_path} --name {setup} --out {out} {meta_args} {coord_arg}
    del model; torch.cuda.empty_cache()

# Sau khi train_all() xong, score E1–E3 bằng score_run(...).


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 ablation: rank là biến được chọn. So sánh r=4 và r=64 trên setup tốt nhất E1–E3.
# Chọn setup_best theo kết quả validation trước khi đánh giá test; không xem test để chọn mô hình.
def run_e4_rank(setup_best):
    results = {}
    for rank in (4, 64):
        run_dir, run = train_experiment(setup_best, rank=rank)
        model = load_trained(run_dir, setup_best)
        preds = predict_rows(model, splits["test"])
        pred_path = f"work/e4_rank_{rank}.jsonl"; write_jsonl(pred_path, preds)
        out = f"report/evidence/e4_rank_{rank}.json"
        meta_args = f"--meta trainable_params={run['trainable_params']} --meta peak_vram_gb={run['peak_vram_gb']:.5f} --meta train_minutes={run['train_minutes']:.5f}"
        !python scripts/score.py --pred {pred_path} --name E4_rank_{rank} --out {out} {meta_args}
        results[rank] = json.loads(Path(out).read_text(encoding="utf-8"))
        del model; torch.cuda.empty_cache()
    return results

# E5 là tuỳ chọn; reward IoU và kiểm thử thuần Python có thể bổ sung sau E0–E4.


## 9. Chạy E1–E4 và chấm kết quả
Chạy cell kế tiếp sau khi E0 và dữ liệu đã sẵn sàng.

In [ ]:
# Run E1-E3, chọn setup bằng validation, rồi chấm test và chạy E4 (rank).
# Mỗi setup có checkpoint adapter mỗi 25 optimizer steps trong /kaggle/working/lab/work.
import subprocess, sys
runs = train_all()
val_scores = {}
for setup, (run_dir, run) in runs.items():
    model = load_trained(run_dir, setup)
    preds = predict_rows(model, splits["val"])
    pred_path = f"work/{setup.lower()}_val.jsonl"
    write_jsonl(pred_path, preds)
    out_path = f"work/{setup.lower()}_val_score.json"
    subprocess.run([sys.executable, "scripts/score.py", "--pred", pred_path, "--split", "val", "--name", setup, "--out", out_path], check=True)
    val_scores[setup] = json.loads(Path(out_path).read_text(encoding="utf-8"))["micro"]["f1"]
    del model; torch.cuda.empty_cache()
print("validation F1:", val_scores)
setup_best = max(val_scores, key=val_scores.get)
print("E4 chosen from validation:", setup_best)
for setup, (run_dir, _) in runs.items():
    score_run(run_dir, setup)
print("E4 rank ablation:", run_e4_rank(setup_best))